In [1]:
import instructor
from enum import Enum
from openai import AzureOpenAI
from functools import partial
from Entity_test import Demographic, Refractory_Relapse, Blast, Diagnosis, test
import pandas as pd
from datetime import datetime
from typing import Optional, List, TypeVar, Type
from pydantic import BaseModel, Field, model_validator, ValidationInfo, ConfigDict
import re
import os


In [2]:
# Define your Azure OpenAI configuration
model = "gpt4o_cursor"
engine = "gpt4o_cursor"
endpoint = "https://gpt4o-xj.openai.azure.com/"

api_version = "2024-02-15-preview"
api_type = "azure"

# key_name = "AZURE_KEY"
api_key = 'YOUR-API-KEY'

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_version=api_version,
    api_key=api_key,
    azure_endpoint=endpoint,
    azure_deployment=engine
)

# Patch the client with instructor
client = instructor.patch(client)

# Optionally, create a partial function with default parameters
in_client = partial(
    client.chat.completions.create,
    model=model,
    temperature=0,
    max_retries=2  # Retries in case of API issues
)


In [3]:
input_dir = '../input'
df = pd.DataFrame(columns=['File Name', 'Inclusion Criteria', 'Exclusion Criteria'])
abstract_list = [] 

for file_name in os.listdir(input_dir):
    if file_name.endswith('.txt'):
        file_path = os.path.join(input_dir, file_name)
        with open(file_path, 'r', encoding='utf-8') as file:
            content = file.read()
            ### Split the notes to inclusion and exclusion criteria seperately
            inclusion_criteria = content.split('Exclusion Criteria:')[0].split('Inclusion Criteria:')[-1].strip()
            exclusion_criteria = content.split('Exclusion Criteria:')[-1].strip()
            new_row = pd.DataFrame({'File Name': [file_name], 'Inclusion Criteria': [inclusion_criteria], 'Exclusion Criteria': [exclusion_criteria]})
            df = pd.concat([df, new_row], ignore_index=True)

abstract_list = df[['File Name', 'Inclusion Criteria', 'Exclusion Criteria']].values.tolist()
print("Abstracts have been successfully read into a list.")

Abstracts have been successfully read into a list.


In [4]:
T = TypeVar("T", bound=BaseModel)

class InfectionClassifier:
    def __init__(self, response_model: Type[T], model_name: str, client):
        self.response_model = response_model
        self.model_name = model_name
        self.client = client

    def ask_ai(self, criteria:str, type:str) -> T:
        return self.client(
            response_model=self.response_model,
            messages=[
                {
                    "role": "system",
                    "content": "You will help extract entities from clinical protocol documents",
                },
                {
                    "role": "user",
                    "content": f"Extract information from the clinical trial eligibility document ({type} criteria).",
                },
                {
                    "role": "user",
                    "content": f"Here is the {type} Criteria section {criteria}",
                },
                {
                    "role": "user",
                    "content": "Please ensure you distinguish between 'CYP3A4' and 'CYP3A'. They are distinct, and an instance of 'CYP3A' should not be interpreted as 'CYP3A4'.",
                },
            ],
            validation_context={"text_chunk": criteria},
        ) 

    def output(self, criteria: str, type: str):
        try:
            response = self.ask_ai(criteria, type)
            entity_dict = response.dict()

            for key, value in entity_dict.items():
                if isinstance(value, list):
                    if value and isinstance(value[0], Enum):
                        entity_dict[key] = [v.value for v in value]  # Extract the string representation
                    # else: keep the list as is
                elif isinstance(value, Enum):
                    entity_dict[key] = value.value  # For single Enum fields

            entity_value_data = {
                "Entity": list(entity_dict.keys()),
                "Value": list(entity_dict.values()),
            }
            df = pd.DataFrame(entity_value_data)
            return df
        
        except Exception as e:
            print(f"Error in output method: {str(e)}")
            # Return an empty DataFrame if an error occurs
            return pd.DataFrame(columns=["Entity", "Value"])

In [5]:
# IC = """
# Inclusion Criteria:\n\n* Participants must have a diagnosis of AML or ALAL and meet the criteria below:\n\n  * Refractory leukemia, defined as persistent leukemia after at least two courses of induction chemotherapy, OR\n  * Early relapsed leukemia, defined as the re-appearance of leukemia after the achievement of remission and within one year of diagnosis, OR\n  * Relapsed leukemia that is refractory to at least one course of salvage therapy (i.e., therapy given after the relapse has occurred), OR\n  * Relapsed leukemia following HCT, OR\n  * Second or greater relapse\n  * Patients with late first relapses, defined as the re-appearance of leukemia after the achievement of remission and greater than one year of diagnosis, may be enrolled in the dose expansion portion of the study after safety data from the dose escalation portion is available.\n\nPatients must have ≥ 5% blasts in the bone marrow as assessed by morphology or flow cytometry. However, if flow cytometry cannot be performed or if an adequate bone marrow sample cannot be obtained (e.g., in a patient with acute megakaryoblastic leukemia with marrow fibrosis), patients may be enrolled if there is unequivocal evidence of leukemia with ≥ 5% blasts in the blood.\n\nIn addition, patients in all categories must not be eligible to undergo curative therapy, such as immediate HCT, because of disease burden, time to identify a stem cell donor, or other reasons.\n\n* Adequate organ function defined as the following:\n\n  * Direct bilirubin ≤ 1.5 x institutional upper limit of normal (ULN)\n  * Normal creatinine for age or a calculated creatinine clearance ≥ 30 mL/min/1.73m\\^2\n  * Left ventricular ejection fraction ≥ 40% or shortening fraction ≥ 25%\n* Patients must be ≥ 2 years of age and ≤ 30 years old. The upper age limit may be defined by each institution, but may not exceed 30 years. Patients treated at St. Jude Children's Research Hospital must be ≤ 24 years old.\n* Performance status: Lansky ≥ 50 for patients who are ≤ 16 years old and Karnofsky ≥ 50% for patients who are \\> 16 years old.\n* At least 14 days must have elapsed since the completion of myelosuppressive therapy or hypomethylating agents and the first doses of venetoclax and selinexor.\n* At least 24 hours must have elapsed since the completion of low-dose or non- myelosuppressive therapy, such as hydroxyurea or low-dose cytarabine (up to 100 mg/m\\^2/day), or leukapheresis, and the first doses of venetoclax and selinexor.\n* For patients who have received prior HCT, there can be no evidence of GVHD and greater than 60 days must have elapsed since the HCT.\n* At least 14 days must have elapsed since the completion of any calcineurin inhibitors (e.g. tacrolimus, cyclosporine).\n* Patients may not receive strong or moderate CYP3A inducers, such as rifampin, within 3 days of the first dose of venetoclax or during the administration of venetoclax. During the dose-escalation portion of the trial, we discourage the use of strong CYP3A inhibitors (e.g., ketoconazole, itraconazole, voriconazole, posaconazole) within 3 days of the first dose of venetoclax or during the administration of venetoclax. However, if an azole is required for the treatment or prevention of fungal infection during any phase of the trial, venetoclax dosing will be reduced to 60 mg/m\\^2 (100 mg max) in patients who require treatment with voriconazole and reduced to 40 mg/m\\^2 (70 mg max) in patients who require posaconazole.
# """  
# IC = """
# Inclusion Criteria:\n\n* Pediatric patients with diagnosis of CD33 positive (\\> 3%),\n\n  * Newly diagnosed secondary AML\n  * Relapsed refractory acute myeloid leukemia by World Health Organization (WHO) criteria Patients must have \\>= 5% blasts in the bone marrow as assessed by morphology or flow cytometry. However, if an adequate bone marrow sample cannot be obtained, patients may be enrolled if there is unequivocal evidence of leukemia with \\>= 5% blasts in the peripheral blood\n* Pediatric Patients with myelodysplastic syndrome (MDS) who progress to AML are eligible at the time of diagnosis of AML regardless of any prior therapy for MDS\n* Performance status: Lansky \\>= 50 for patients who are =\\< 16 years old and Karnofsky \\>= 50% for patients who are \\> 16 years old\n* Age =\\< 21 years of age\n* Total serum bilirubin =\\< 2 x upper limit of normal (ULN). Patients with known Gilbert's syndrome may have a total bilirubin up to =\\< 3 x ULN\n* Serum creatinine =\\< 2.0 mg/dl\n* Alanine aminotransferase (ALT) and/or aspartate aminotransferase (AST) =\\< 3 x ULN; =\\< 5 x ULN in case of suspected leukemic liver involvement\n* Females of childbearing potential must have a negative serum or urine beta human chorionic gonadotropin (beta-HCG) pregnancy test result within 14 days prior to the first dose of study drugs and must agree to use one of the following effective contraception methods during the study and for 30 days following the last dose of study drug. Effective methods of birth control include:\n\n  * Birth control pills, skin patches, shots, implants (placed under the skin by a health care provider)\n  * Intrauterine devices (IUDs)\n  * Condom or occlusive cap (diaphragm or cervical/vault caps) used with Spermicide\n  * Abstinence\n* Males, need to inform the doctor right away if the partner becomes pregnant or suspects pregnancy. While in this study and for 30 days after the last treatment the patient should not donate sperm for the purposes of reproduction. He will need to use a condom while in this study and for 30 days after the last treatment\n\n
# """
# IC = """
# Inclusion Criteria:\n\n* Prior morphologically confirmed diagnosis of AML based on WHO Criteria\n* Has previously failed all available and suitable therapies for AML Disease relapse or the presence of refractory disease after ≥ 2 cycles of chemotherapy must be documented by bone marrow (BM) examination demonstrating \\> 5% blasts in the BM not attributable to another cause. Administration of hydroxyurea 10 to 20 mg/kg/day PO (maximum 1000 mg PO BID) to control high WBC count is permitted.\n* Age ≥ 2 and ≤ 25 years\n* Body surface area (BSA) ≤ 2.10 m2, calculated per the Mostellar formula\n* Must be able to tolerate po or ng medications.\n* Performance status:\n* Subject age\n\n  ≤ 16 years old: Lansky score ≥ 50 \\> 16 years old: Karnofsky score ≥ 50%\n* Life expectancy of greater than 4 weeks\n* Platelets ≥ 10,000/mm3 (for subjects with platelets \\< 10,000/mm3 at baseline, platelet transfusion support is allowed)\n* Serum creatinine ≤ 2.0 mg/dL or estimated creatinine clearance ≥ 30 mL/min (Cockcroft Gault) within 14 days prior to treatment initiation\n* Total bilirubin ≤ 2.0 x Institutional upper limit of normal (ULN) within 14 days prior to treatment initiation, unless the elevation is due primarily to elevated unconjugated hyperbilirubinemia secondary to Gilbert's syndrome or hemolysis or non hepatic origin, and not to liver dysfunction\n* SGOT (AST) ≤ 3.0 x ULN and SGPT (ALT) ≤ 3.0 x ULN within 14 days prior to treatment initiation\n* Females of reproductive potential (WOCBP) must have a negative pregnancy test within 14 days prior to study treatment.\n* WOCBP must agree to use adequate contraception (eg, hormonal or barrier methods of birth control; abstinence; sterilized partner) for the duration of study participation\n* Men only: Men must agree to use adequate contraception (eg, hormonal or barrier methods of birth control; abstinence; sterilized partner) prior to the study treatment (from date of consent), for the duration of study participation, and 30 days after completion of niclosamide administration\n* Ability to understand the purpose and risks of the study and the willingness to sign a written informed consent document containing an authorization to use protected health information (in accordance with national and local subject privacy regulations\n\n
# """
# EC = """
# Exclusion Criteria:\n\n* Received anticancer therapy (chemotherapy, immunotherapy, radiotherapy, or investigational therapy) within 2 weeks prior to starting study treatment. Administration of hydroxyurea 10 to 20 mg/kg/day PO (maximum 1000 mg PO BID) to control high WBC is permitted.\n* Receiving any other investigational agents.\n* Unresolved toxicities due to prior anticancer therapy, defined as not having resolved to Grade 0 or 1 (by CTCAE version 5 criteria), unless otherwise defined in the inclusion/exclusion criteria with the exception of alopecia\n* Acute promyelocytic leukemia (French American British Class M3 AML)\n* Known active central nervous system (CNS) leukemia; subjects can enroll on study if CNS disease can be cleared with intrathecal chemotherapy, in the judgement of the treating physician\n* Prior bone marrow transplant presenting with active uncontrolled graft versus host disease (GvHD)\n* Known congenital bleeding disorders, including but not limited to hemophilia\n* Known active uncontrolled systemic infection\n* Malabsorption syndrome, disease significantly affecting gastrointestinal function, or resection of the stomach or small bowel, uncontrolled symptomatic inflammatory bowel disease or ulcerative colitis, or partial or complete bowel obstruction, at the time of study entry\n* Inability to receive administration of niclosamide in the available formulation(s)\n* Uncontrolled intercurrent illness including, but not limited to, uncontrolled active infection, or psychiatric illness/social situations that would limit compliance with study requirements\n* Lactating or pregnant female\n* Known active hepatitis C
# """

In [6]:
# response_models = [Demographic, Refractory_Relapse, Blast, Diagnosis]  # List of response models you have
# for item in abstract_list:
#     file_name, inclusion_criteria, exclusion_criteria = item  
#     clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
#     clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
#     entity_value_list = []
#     # Iterate over each response_model
#     for response_model in response_models:
#         # Create an instance of the InfectionClassifier for each model
#         Classifier = InfectionClassifier(response_model=response_model, model_name=model, client=in_client)
        
#         # Get the inclusion and exclusion criteria as DataFrames
#         df_IC = Classifier.output(criteria=clean_text_ic, type='Inclusion')
#         df_EC = Classifier.output(criteria=clean_text_ec, type='Exclusion')
        
#         # Merge the two DataFrames on the 'Entity' column
#         df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))
        
#         # Append the resulting DataFrame to the list
#         entity_value_list.append(df_final)

#     # Concatenate all DataFrames in the list into a single DataFrame
#     df_concatenated = pd.concat(entity_value_list, ignore_index=True)
#     output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
#     df_concatenated.to_excel(output_excel_path, index=False)

In [7]:
response_models = [test]
file_name, inclusion_criteria, exclusion_criteria = abstract_list[0]  
clean_text_ic = inclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
clean_text_ec = exclusion_criteria.replace("  ", " ").replace("\n", "; ").replace(';',' ')
entity_value_list = []
# Iterate over each response_model
for response_model in response_models:
    # Create an instance of the InfectionClassifier for each model
    Classifier = InfectionClassifier(response_model=response_model, model_name=model, client=in_client)
    
    # Get the inclusion and exclusion criteria as DataFrames
    df_IC = Classifier.output(criteria=clean_text_ic, type='Inclusion')
    df_EC = Classifier.output(criteria=clean_text_ec, type='Exclusion')
    
    # Merge the two DataFrames on the 'Entity' column
    df_final = pd.merge(df_IC, df_EC, on='Entity', how='outer', suffixes=('_IC', '_EC'))
    
    # Append the resulting DataFrame to the list
    entity_value_list.append(df_final)

# Concatenate all DataFrames in the list into a single DataFrame
df_concatenated = pd.concat(entity_value_list, ignore_index=True)
# output_excel_path = f'../results/{file_name}_output_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.xlsx'
# df_concatenated.to_excel(output_excel_path, index=False)

In [ ]:
df_concatenated